In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../output/transactions_data_non_nulls.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10871 entries, 0 to 10870
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             10871 non-null  object 
 1   customer_id                10871 non-null  object 
 2   timestamp                  10871 non-null  object 
 3   home_country               10871 non-null  object 
 4   source_currency            10871 non-null  object 
 5   dest_currency              10871 non-null  object 
 6   channel                    10871 non-null  object 
 7   amount_src                 10871 non-null  float64
 8   amount_usd                 10871 non-null  float64
 9   fee                        10871 non-null  float64
 10  exchange_rate_src_to_dest  10871 non-null  float64
 11  device_id                  10871 non-null  object 
 12  new_device                 10871 non-null  bool   
 13  ip_address                 10871 non-null  obj

### Drop Irrelevant columns

In [4]:
cols_to_drop = ['transaction_id', 'customer_id', 'device_id', 'ip_address']

df = df.drop(columns=cols_to_drop)

**Justification**:

Upon reviewing the data, we would drop only the Transactions_ID column rather than dropping all columns that have IDs. The reasoning behind this is that some columns like customerID, deviceID are meaningful to the context/scope of the project. We may be able to discern the unique interactions of a specific customer through the customerID column. Therefore, any change in the determined pattern may be grounds to suspect fraud. 

### Conversion of timestamp column
We would begin our cleaning by converting the timestamp column to a datetime object.

In [5]:
# df['timestamp'] = pd.to_datetime(df['timestamp'])

In [6]:
# df.iloc[39]

**Insight**

Row 39 contains an invalid date '2025/13/40'. Since no other is number '13' and no day is numbered '40'. We would go ahead and drop it.

Row 200 contains an invalid date as well "0000-00-00T00:00:00Z". This was handled similarly to row 39.

In [7]:
# df.drop(index=39, axis=0, inplace=True)
# df.reset_index(drop=True, inplace=True)

In [8]:
# df['timestamp'] = pd.to_datetime(df['timestamp'])

In [9]:
# df.iloc[201]

In [10]:
# df.drop(index=200, axis=0, inplace=True)
# df.reset_index(drop=True, inplace=True)
# df['timestamp'] = pd.to_datetime(df['timestamp'])

It can be noticed that there is a pattern forming where the formats of the dates are invalid i.e. they are being written as "0000-00-00T00:00:00Z".

we would therefore filter the dataset to uncover all dates in that format

In [11]:
# df[df['timestamp'] == "0000-00-00T00:00:00Z"]

There is a total of 20 rows that have their dates in this particular format. 

We now filter out the dates that are appear in this format and reassign the new data to the 'df' variable.

In [12]:
# # Select all rows that are not in the specified format
# df =df[df['timestamp'] != "0000-00-00T00:00:00Z"]

# # Resume converting the timestamp column
# df['timestamp'] = pd.to_datetime(df['timestamp'])

In [13]:
# df.drop(index=39, axis=0, inplace=True)
# df.reset_index(drop=True, inplace=True)
# df['timestamp'] = pd.to_datetime(df['timestamp'])

# df.drop(index=200, axis=0, inplace=True)
# df.reset_index(drop=True, inplace=True)
# df['timestamp'] = pd.to_datetime(df['timestamp'])

# # Resume converting the timestamp column
# df['timestamp'] = pd.to_datetime(df['timestamp'])

In [14]:
# Select all rows that are not in the specified format
df =df[df['timestamp'] != "0000-00-00T00:00:00Z"]

# Select all rows that are not in the specified format
df = df[df['timestamp'] != "2025/13/40 25:61:00"]

df['timestamp'] = pd.to_datetime(df['timestamp'])

This is the second time is particular date is appearing. Let's filter and determine if there are others rows with this format in the df.

In [15]:
# df[df['timestamp'] == "2025/13/40 25:61:00"]

There are 9 rows with this invalid date formats that would have to be cleaned.

In [16]:
# # Select all rows that are not in the specified format
# df = df[df['timestamp'] != "2025/13/40 25:61:00"]

In [17]:
# # Resume converting the timestamp column
# df['timestamp'] = pd.to_datetime(df['timestamp'])

In [18]:
# df.info()

### Extracting time values from timestamp column

In [19]:
df['timestamp'].describe()[['min', 'max']]

min    2022-10-03 18:40:59.468549+00:00
max    2025-12-16 00:13:41.468549+00:00
Name: timestamp, dtype: object

In [20]:
df['year'] = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df['day'] = df['timestamp'].dt.day
df['hour'] = df['timestamp'].dt.hour
df['minute'] = df['timestamp'].dt.minute

# dropping timestamp after use
df = df.drop(columns='timestamp')

### Column wise Checks

The next step would be to perform a column wise check to verify if there is any typographical error in each columns.

In [21]:
cat_cols = df.select_dtypes(include='object')

for col in cat_cols:
    if not df[col].name == 'customer_id' and not df[col].name == 'device_id' and not df[col].name == 'ip_address':
        print(f'{df[col].name}: {df[col].nunique()} unique values')
        print(f'{df[col].name}: {df[col].unique()}')
        print('\n')

home_country: 4 unique values
home_country: ['US' 'CA' 'UK' 'unknown']


source_currency: 3 unique values
source_currency: ['USD' 'CAD' 'GBP']


dest_currency: 9 unique values
dest_currency: ['CAD' 'MXN' 'CNY' 'EUR' 'INR' 'GBP' 'PHP' 'NGN' 'USD']


channel: 9 unique values
channel: ['ATM' 'web' 'mobile' 'WEB' 'MOBILE' 'unknown' 'mobille' 'weeb' 'ATm']


ip_country: 4 unique values
ip_country: ['US' 'CA' 'UK' 'unknown']


kyc_tier: 9 unique values
kyc_tier: ['standard' 'enhanced' 'low' 'standrd' 'STANDARD' 'unknown' 'enhancd'
 'ENHANCED' 'LOW']




**Insights**

There are typographical errors in the data
- channel columns contains errors e.g. 'web', 'WEB'
- kyc_tier columns contains errors e.g. 'standard', 'standrd'

There is also inconsistent formatting of string values
- 'unknown' is lowwer case in some columns but appearing in upper case ('UNKNOWN')  

In [22]:
# Correcting inconsistency in channel column
df['channel'] = df['channel'].replace({'web': 'WEB', 'weeb':'WEB'},)
df['channel'] = df['channel'].replace({'ATm': 'ATM'})
df['channel'] = df['channel'].replace({'mobile': 'MOBILE', 'mobille':'MOBILE'},)

# Correcting inconsistency in kyc_tier column
df['kyc_tier'] = df['kyc_tier'].replace({'standrd': 'standard'},)
df['kyc_tier'] = df['kyc_tier'].replace({'enhancd': 'enhanced'},)

In [23]:
# Standardizing the capitialization of text in the data
for col in cat_cols:
    df[col] = df[col].str.upper()

In [24]:
# Comfirmation of changes by repeating the entire process
cat_cols = df.select_dtypes(include='object')

for col in cat_cols:
    if not df[col].name == 'customer_id' and not df[col].name == 'device_id' and not df[col].name == 'ip_address':
        print(f'{df[col].name}: {df[col].nunique()} unique values')
        print(f'{df[col].name}: {df[col].unique()}')
        print('\n')

home_country: 4 unique values
home_country: ['US' 'CA' 'UK' 'UNKNOWN']


source_currency: 3 unique values
source_currency: ['USD' 'CAD' 'GBP']


dest_currency: 9 unique values
dest_currency: ['CAD' 'MXN' 'CNY' 'EUR' 'INR' 'GBP' 'PHP' 'NGN' 'USD']


channel: 4 unique values
channel: ['ATM' 'WEB' 'MOBILE' 'UNKNOWN']


ip_country: 4 unique values
ip_country: ['US' 'CA' 'UK' 'UNKNOWN']


kyc_tier: 4 unique values
kyc_tier: ['STANDARD' 'ENHANCED' 'LOW' 'UNKNOWN']




**Result**

- No typographical errors
- Uniform capitialization of all text values

In [25]:
df.columns

Index(['home_country', 'source_currency', 'dest_currency', 'channel',
       'amount_src', 'amount_usd', 'fee', 'exchange_rate_src_to_dest',
       'new_device', 'ip_country', 'location_mismatch', 'ip_risk_score',
       'kyc_tier', 'account_age_days', 'device_trust_score',
       'chargeback_history_count', 'risk_score_internal', 'txn_velocity_1h',
       'txn_velocity_24h', 'corridor_risk', 'is_fraud', 'year', 'month', 'day',
       'hour', 'minute'],
      dtype='object')

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10840 entries, 0 to 10870
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   home_country               10840 non-null  object 
 1   source_currency            10840 non-null  object 
 2   dest_currency              10840 non-null  object 
 3   channel                    10840 non-null  object 
 4   amount_src                 10840 non-null  float64
 5   amount_usd                 10840 non-null  float64
 6   fee                        10840 non-null  float64
 7   exchange_rate_src_to_dest  10840 non-null  float64
 8   new_device                 10840 non-null  bool   
 9   ip_country                 10840 non-null  object 
 10  location_mismatch          10840 non-null  bool   
 11  ip_risk_score              10840 non-null  float64
 12  kyc_tier                   10840 non-null  object 
 13  account_age_days           10840 non-null  int64  


### Removing transactions under $0

In [27]:
df[df['amount_src'] < 0]

,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,exchange_rate_src_to_dest,new_device,ip_country,...,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud,year,month,day,hour,minute
314,US,USD,NGN,MOBILE,-37.08,37.08,9999.99,1111.111111,False,US,...,0.166,-1,0,0.25,0,2022,11,10,3,14
537,US,USD,USD,WEB,-176.92,176.92,9999.99,1.000000,False,US,...,0.087,-1,0,0.00,0,2022,12,7,17,24
549,US,USD,CAD,MOBILE,-702.71,702.71,9999.99,1.351351,False,US,...,0.176,-1,0,0.00,0,2022,12,8,16,48
568,US,USD,GBP,MOBILE,-230.36,230.36,9999.99,0.800000,True,US,...,0.223,-1,0,0.00,0,2023,4,28,13,36
574,CA,CAD,CAD,WEB,-203.83,150.83,9999.99,1.000000,False,CA,...,0.268,-1,0,0.00,0,2022,12,11,19,23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9472,UK,GBP,USD,WEB,-69.26,86.58,9999.99,1.250000,False,UK,...,0.407,-1,1,0.00,0,2025,9,11,20,23
9534,US,USD,USD,WEB,-190.00,190.00,9999.99,1.000000,False,US,...,0.247,-1,0,0.00,0,2025,9,18,4,31
9550,UK,GBP,CNY,MOBILE,-245.82,307.28,9999.99,8.928571,False,UK,...,0.407,-1,0,0.00,0,2025,9,19,23,56
9567,UK,GBP,EUR,MOBILE,-186.29,232.86,9999.99,1.157407,False,UK,...,0.169,-1,0,0.05,0,2025,9,22,5,32


In [28]:
df[df['amount_src'] < 0]['fee']

314     9999.99
537     9999.99
549     9999.99
568     9999.99
574     9999.99
         ...   
9472    9999.99
9534    9999.99
9550    9999.99
9567    9999.99
9636    9999.99
Name: fee, Length: 97, dtype: float64

In [29]:
df = df[df['amount_src'] > 0]

### Removing device_trust_score under 0

In [30]:
with pd.option_context('display.max_columns', None):
    display(df[df['device_trust_score'] < 0].head())

,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,exchange_rate_src_to_dest,new_device,ip_country,location_mismatch,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud,year,month,day,hour,minute
49,US,USD,EUR,WEB,67.19,67.19,-1.0,0.925926,False,CA,True,1.2,STANDARD,263,-0.1,0,0.223,-1,0,0.0,0,2022,10,9,8,57
58,US,USD,USD,MOBILE,190.42,190.42,-1.0,1.000000,False,US,False,1.2,STANDARD,298,-0.1,0,0.166,-1,0,0.0,0,2022,10,10,12,6
76,US,USD,INR,WEB,130.13,130.13,-1.0,83.333333,False,US,False,1.2,ENHANCED,257,-0.1,0,0.257,-1,0,0.0,0,2022,10,12,8,33
215,US,USD,GBP,ATM,81.19,81.19,-1.0,0.800000,False,US,False,1.2,STANDARD,147,-0.1,0,0.391,-1,1,0.0,0,2022,10,28,19,29
334,US,USD,CAD,WEB,316.74,316.74,-1.0,1.351351,False,US,False,1.2,STANDARD,298,-0.1,0,0.166,-1,0,0.0,0,2022,11,12,23,11


In [31]:
under_neg_1 = df[df['device_trust_score'] < 0]

print(under_neg_1['device_trust_score'].unique())
print(under_neg_1['fee'].unique())

[-0.1]
[-1.]


In [32]:
df = df[df['device_trust_score'] > 0]

### Export final result

In [33]:
df.to_csv('../output/transactions_data_completely_clean.csv', index=False)